# UCM GPS Quality Comparison - Unfiltered vs Filtered vs Adjusted

Compares GPS trajectories from the UCM backpack across 5 experiment phases.
Data is clipped to the exact phase time windows from `key.csv`.

**Report 1**: Raw (unfiltered) 1-sec GPS **vs** after HDOP/IO_flag filter  
**Report 2**: HDOP/IO_flag filtered **vs** after 10-sec median resampling

In [ ]:
# ===========================================================================
# CONFIG - Paths, constants, colour palettes
# ===========================================================================

import warnings; warnings.filterwarnings('ignore')
from pathlib import Path
import numpy as np
import pandas as pd

# -- Paths --------------------------------------------------------------------
RAW_UCM_DIR  = Path(r'C:\Users\pandya\Documents\Github\docker\Paper3_Github\rawdata\02_ucm')
KEY_FILE     = Path(r'C:\Users\pandya\Documents\Github\docker\Paper3_Github\output\key.csv')

# -- Phases to plot -----------------------------------------------------------
PHASES = ['BikeU', 'WalkU', 'BikeG', 'WalkG', 'Tram']

# -- PARTICIPANT colours - P2 to P18, 17 distinct saturated colours ----------
PARTICIPANT_COLORS = {
    'P2':  '#e63946',   # vivid red
    'P3':  '#2a9d8f',   # teal
    'P4':  '#264653',   # dark slate
    'P5':  '#e76f51',   # burnt sienna
    'P6':  '#457b9d',   # steel blue
    'P7':  '#6a4c93',   # purple
    'P8':  '#1d3557',   # dark navy
    'P9':  '#f4a261',   # sandy orange
    'P10': '#2d6a4f',   # forest green
    'P11': '#c1121f',   # dark red
    'P12': '#780000',   # maroon
    'P13': '#3a86ff',   # bright blue
    'P14': '#8338ec',   # violet
    'P15': '#fb5607',   # vivid orange-red
    'P16': '#005f73',   # dark teal
    'P17': '#606c38',   # olive green
    'P18': '#bc6c25',   # warm brown
}

# -- PHASE colours (for section backgrounds) ---------------------------------
PHASE_COLORS = {
    'BikeU': '#d45500',
    'WalkU': '#b8860b',
    'BikeG': '#1a6b1a',
    'WalkG': '#52b852',
    'Tram':  '#7f8c8d',
}

print("OK  Config loaded")
print(f"    Raw UCM files: {RAW_UCM_DIR}")
print(f"    Participants: {sorted(PARTICIPANT_COLORS.keys())}")
print(f"    Phases: {PHASES}")


In [ ]:
# ===========================================================================
# LOAD - Read key.csv, build phase windows, read raw 1-sec UCM CSVs
# ===========================================================================

def _parse_ucm_columns(path: Path):
    """Extract column names from the '# GPS_time,...' comment line."""
    with open(path, 'r', errors='replace') as f:
        for line in f:
            stripped = line.lstrip('# ').strip()
            if stripped.startswith('GPS_time'):
                return [c.strip() for c in stripped.split(',')]
    return None


def read_raw_ucm_csv(path: Path):
    """Read a raw 1-sec UCM data.csv -> DataFrame with parsed columns & datetime."""
    col_names = _parse_ucm_columns(path)
    if col_names is None:
        return None
    df = pd.read_csv(path, comment='#', header=None, names=col_names, low_memory=False)
    df['GPS_time'] = pd.to_datetime(df['GPS_time'], errors='coerce')
    for col in ['GPS_lat', 'GPS_lon', 'GPS_HDOP', 'IO_flag']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['GPS_time', 'GPS_lat', 'GPS_lon']).sort_values('GPS_time').reset_index(drop=True)
    return df


def build_phase_windows(key: pd.DataFrame) -> dict:
    """
    Return {(pid_int, phase_str): (start_ts, end_ts)} in Brussels local time.
    Key.csv stores UTC times; add 2h for CEST (Brussels summer).
    """
    windows = {}
    for _, row in key.iterrows():
        pid = int(row['Participant_ID'])
        date_str = pd.to_datetime(str(row['Date']) + '-2025',
                                  format='%d-%b-%Y').strftime('%Y-%m-%d')
        for ph in PHASES:
            s_col, e_col = f'{ph}_start', f'{ph}_end'
            if s_col not in row or pd.isna(row.get(s_col)) or pd.isna(row.get(e_col)):
                continue
            start = pd.Timestamp(f"{date_str} {row[s_col]}") + pd.Timedelta(hours=2)
            end   = pd.Timestamp(f"{date_str} {row[e_col]}") + pd.Timedelta(hours=2)
            if end < start:
                end += pd.Timedelta(days=1)
            windows[(pid, ph)] = (start, end)
    return windows


# -- Read key.csv and build phase windows ------------------------------------
key = pd.read_csv(KEY_FILE)
phase_windows = build_phase_windows(key)
print(f"OK  Phase windows built: {len(phase_windows)} windows")

# -- Read ALL raw UCM files -> { (pid, phase): DataFrame } --------------------
raw_data = {}
csv_files = sorted(RAW_UCM_DIR.glob('P*_*.csv'))
for csv_path in csv_files:
    stem = csv_path.stem  # e.g. "P10_BikeU"
    pid_str, phase = stem.split('_')
    pid = int(pid_str[1:])
    df = read_raw_ucm_csv(csv_path)
    if df is None or len(df) < 4:
        continue

    # -- Shift raw timestamps from UTC -> Brussels (+2h) if needed ----------
    # The UCM records in UTC; if the first timestamp is ~2h before the phase
    # window start (in Brussels time), shift forward by 2h.
    if (pid, phase) in phase_windows:
        win_start, win_end = phase_windows[(pid, phase)]
        offset_h = (win_start - df['GPS_time'].iloc[0]).total_seconds() / 3600
        if 1.5 < offset_h < 2.5:
            df['GPS_time'] = df['GPS_time'] + pd.Timedelta(hours=2)
            print(f"    [UTC->CEST] P{pid} {phase}: shifted +2h")

        # -- Clip to phase window ------------------------------------------
        before = len(df)
        df = df[(df['GPS_time'] >= win_start) & (df['GPS_time'] <= win_end)].copy()
        clipped = before - len(df)
        if clipped > 0:
            print(f"    [clip] P{pid} {phase}: removed {clipped} rows outside window")

    raw_data[(pid, phase)] = df

print(f"\nOK  Loaded {len(raw_data)} participant-phase files")
for ph in PHASES:
    n = sum(1 for (p, phase) in raw_data if phase == ph)
    print(f"    {ph}: {n} participants")

In [ ]:
# ===========================================================================
# DEFINE - Filtering, resampling, and matplotlib plotting helpers
# ===========================================================================

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

def flag_bad_gps(df: pd.DataFrame) -> np.ndarray:
    """
    Boolean mask: True = bad GPS point (should be excluded).
    Criteria: HDOP > 5  OR  IO_flag == 9.
    """
    n = len(df)
    bad = np.zeros(n, dtype=bool)
    if 'GPS_HDOP' in df.columns:
        bad |= (df['GPS_HDOP'].values > 5)
    if 'IO_flag' in df.columns:
        bad |= (df['IO_flag'].values == 9)
    bad[1:] |= bad[:-1].copy()
    return bad


def resample_10sec_median(df: pd.DataFrame) -> pd.DataFrame:
    """Resample 1-sec GPS -> 10-sec bins using median."""
    df = df.set_index('GPS_time').sort_index()
    num_cols = df.select_dtypes(include='number').columns.tolist()
    df_10s = df[num_cols].resample('10s').median().reset_index()
    df_10s['GPS_time'] = df_10s['GPS_time'].dt.floor('10s')
    return df_10s


def resample_10sec_5th_sample(df: pd.DataFrame) -> pd.DataFrame:
    """Resample 1-sec GPS -> 10-sec bins by taking the 5th (or middle) sample."""
    df = df.set_index('GPS_time').sort_index()
    bins = df.groupby(pd.Grouper(freq='10s'))
    rows = []
    for ts, grp in bins:
        if len(grp) == 0:
            continue
        idx = min(4, len(grp) // 2)
        row = grp.iloc[idx].copy()
        row.name = ts.floor('10s')
        rows.append(row)
    result = pd.DataFrame(rows)
    result.index.name = 'GPS_time'
    return result.reset_index()


def build_phase_gps(raw_data: dict, phase: str, apply_filter: bool = True,
                    resample_method: str = None) -> dict:
    """Collect GPS data for one phase across all participants."""
    result = {}
    for (pid, ph), df in raw_data.items():
        if ph != phase:
            continue
        pid_key = f'P{pid}'
        d = df[['GPS_time', 'GPS_lat', 'GPS_lon', 'GPS_HDOP', 'IO_flag']].copy()

        if apply_filter:
            bad = flag_bad_gps(d)
            d = d[~bad].copy()

        if resample_method == 'median':
            d = resample_10sec_median(d)
        elif resample_method == '5th_sample':
            d = resample_10sec_5th_sample(d)

        d = d.dropna(subset=['GPS_lat', 'GPS_lon'])
        if len(d) >= 2:
            result[pid_key] = d
    return result


def plot_phase_comparison(phase: str, raw_data: dict,
                          left_label: str, right_label: str,
                          left_filter: bool, right_filter: bool,
                          right_resample: str = None,
                          figsize=(16, 10)):
    """
    Create a matplotlib figure with two subplots:
    Left  = one version of the GPS tracks (e.g. unfiltered)
    Right = another version (e.g. filtered, or median-resampled)
    Each subplot shows all participants as coloured tracks.
    """
    left_gps  = build_phase_gps(raw_data, phase,
                                 apply_filter=left_filter,
                                 resample_method=None)
    right_gps = build_phase_gps(raw_data, phase,
                                 apply_filter=right_filter,
                                 resample_method=right_resample)

    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=figsize)
    fig.suptitle(f'{phase}', fontsize=16, fontweight='bold', y=0.98)

    handles = []
    for pid in sorted(PARTICIPANT_COLORS.keys()):
        # Check if this participant exists in either plot
        in_left  = pid in left_gps
        in_right = pid in right_gps
        if not in_left and not in_right:
            continue
        c = PARTICIPANT_COLORS[pid]
        handles.append(mpatches.Patch(color=c, label=pid))

        for ax, gps_dict, lbl in [
            (ax_left, left_gps, left_label),
            (ax_right, right_gps, right_label)
        ]:
            if pid in gps_dict:
                df = gps_dict[pid]
                ax.plot(df['GPS_lon'], df['GPS_lat'],
                        color=c, linewidth=1.2, alpha=0.8)
                ax.scatter(df['GPS_lon'].iloc[0], df['GPS_lat'].iloc[0],
                           color=c, s=30, zorder=5, marker='o')
                ax.scatter(df['GPS_lon'].iloc[-1], df['GPS_lat'].iloc[-1],
                           color=c, s=30, zorder=5, marker='s')

        ax_left.set_title(f'[raw] {left_label}', fontsize=11, fontweight='bold')
        ax_right.set_title(f'[filtered] {right_label}', fontsize=11, fontweight='bold')

    for ax in [ax_left, ax_right]:
        ax.set_xlabel('Longitude')
        ax.set_ylabel('Latitude')
        ax.grid(True, alpha=0.3)
        ax.set_aspect('equal', adjustable='datalim')

    ax_left.legend(handles=handles, loc='upper left', fontsize=7,
                   title='Participant', title_fontsize=8,
                   ncol=2, framealpha=0.9)

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    return fig


print("OK  All functions defined (matplotlib mode)")

## Raw (unfiltered) 1-sec GPS vs HDOP/IO_flag Filtered

**Left**: raw GPS coordinates as recorded by the UCM backpack (1-sec resolution).  
**Right**: after removing points where HDOP > 5 or IO_flag == 9 (the current build pipeline filter).

Each map has per-participant coloured tracks with toggle checkboxes.

In [ ]:
# ===========================================================================
# Report 1 - Unfiltered vs HDOP/IO_flag Filtered
# One matplotlib figure per phase
# ===========================================================================

print("Report 1: Raw (unfiltered) vs HDOP/IO_flag filtered GPS")
print("=" * 60)

for ph in PHASES:
    print(f"  {ph} ...")
    fig = plot_phase_comparison(
        phase=ph,
        raw_data=raw_data,
        left_label='Raw (unfiltered, 1-sec)',
        right_label='HDOP/IO_flag filtered (1-sec)',
        left_filter=False,
        right_filter=True,
        right_resample=None
    )
    plt.show()
    plt.close(fig)

print("\nOK  All 5 phases plotted above")

## HDOP/IO_flag Filtered (1-sec) vs Median-resampled (10-sec)

**Left**: HDOP/IO_flag filtered GPS at 1-sec resolution (same as previous report right column).  
**Right**: after filtering **plus** resampling to 10-sec bins using the **median** of each bin
(instead of the mean used in `02_ucm_build.py`).  

This addresses the colleague's suggestion:  
> *"could it be better to use the medium lat&lon, or the 5th recording in each 10-sec window-"*

The median is more robust to outlier GPS points within a 10-sec window than the mean.

In [ ]:
# ===========================================================================
# Report 2 - Filtered vs Median-resampled (10-sec)
# One matplotlib figure per phase
# ===========================================================================

print("Report 2: HDOP/IO_flag filtered vs median 10-sec resampled GPS")
print("=" * 60)

for ph in PHASES:
    print(f"  {ph} ...")
    fig = plot_phase_comparison(
        phase=ph,
        raw_data=raw_data,
        left_label='HDOP/IO_flag filtered (1-sec)',
        right_label='Filtered + median 10-sec bins',
        left_filter=True,
        right_filter=True,
        right_resample='median'
    )
    plt.show()
    plt.close(fig)

print("\nOK  All 5 phases plotted above")

## GPS speed & duration per phase

In [ ]:
# ===========================================================================
# GPS speed & duration per phase
# 2x2 subplots + participant summary table
# Speed = cumulative filtered GPS path distance divided by key-based phase duration.
# ===========================================================================

phase_order = ['WalkG', 'BikeG', 'WalkU', 'BikeU']
phase_gps_filtered = {
    ph: build_phase_gps(raw_data, ph, apply_filter=True, resample_method=None)
    for ph in phase_order
}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

def cumulative_path_km(df):
    if len(df) < 2:
        return np.nan
    total_km = 0.0
    for i in range(len(df) - 1):
        total_km += haversine_km(
            df['GPS_lat'].iloc[i], df['GPS_lon'].iloc[i],
            df['GPS_lat'].iloc[i + 1], df['GPS_lon'].iloc[i + 1]
        )
    return total_km

fig, axes = plt.subplots(2, 2, figsize=(15, 12), constrained_layout=True)
axes = axes.ravel()
participant_ids = sorted({f'P{pid}' for (pid, _) in raw_data.keys()}, key=lambda x: int(x[1:]))

for ax, phase in zip(axes, phase_order):
    gps_dict = phase_gps_filtered[phase]
    ax.set_title(phase, fontsize=11, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='datalim')

    phase_distances = []
    phase_speeds = []
    phase_durations = []
    for pid in participant_ids:
        if pid not in gps_dict:
            continue
        df = gps_dict[pid].copy()
        c = PARTICIPANT_COLORS.get(pid, '#333333')
        ax.plot(df['GPS_lon'], df['GPS_lat'], color=c, linewidth=1.2, alpha=0.85)
        ax.scatter(df['GPS_lon'].iloc[0], df['GPS_lat'].iloc[0], color=c, s=18, marker='o', zorder=4)
        ax.scatter(df['GPS_lon'].iloc[-1], df['GPS_lat'].iloc[-1], color=c, s=22, marker='s', zorder=5)

        dist_km = cumulative_path_km(df)
        phase_distances.append(dist_km)
        pid_int = int(pid[1:])
        if (pid_int, phase) in phase_windows:
            start_ts, end_ts = phase_windows[(pid_int, phase)]
            dur_h = (end_ts - start_ts).total_seconds() / 3600.0
            if dur_h > 0:
                phase_speeds.append(dist_km / dur_h)
                phase_durations.append(dur_h * 60.0)

    if phase_distances:
        avg_dist_km = float(np.nanmean(phase_distances))
        avg_speed_kmh = float(np.nanmean(phase_speeds)) if phase_speeds else np.nan
        avg_dur_min = float(np.nanmean(phase_durations)) if phase_durations else np.nan
        ax.text(
            0.52, 0.18,
            f'distance = {avg_dist_km:.2f} km\nspeed = {avg_speed_kmh:.2f} km/h\nduration = {avg_dur_min:.2f} min',
            transform=ax.transAxes,
            fontsize=ax.xaxis.label.get_size(),
            fontweight='bold',
            color='black',
            ha='center', va='center',
            bbox=dict(facecolor='white', edgecolor='black', alpha=0.82, boxstyle='round,pad=0.3')
        )

fig.suptitle('Filtered UCM GPS by phase with cumulative-distance labels',
             fontsize=14, fontweight='bold')
plt.show()

summary_rows = []
for pid in participant_ids:
    pid_int = int(pid[1:])
    row = {'ParticipantID': pid}
    for phase in phase_order:
        gps_dict = phase_gps_filtered[phase]
        speed_col = f'{phase}_speed'
        dur_col = f'{phase}_dur'

        if pid in gps_dict and (pid_int, phase) in phase_windows:
            df = gps_dict[pid]
            dist_km = cumulative_path_km(df)
            start_ts, end_ts = phase_windows[(pid_int, phase)]
            dur_min = (end_ts - start_ts).total_seconds() / 60.0
            speed_kmh = dist_km / (dur_min / 60.0) if dur_min > 0 else np.nan
            row[speed_col] = round(speed_kmh, 3)
            row[dur_col] = round(dur_min, 2)
        else:
            row[speed_col] = np.nan
            row[dur_col] = np.nan
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
display(summary_df)



## Official route files and duration plausibility

BikeG uses the finalized shortened common green-cycling route from `Experiment path/BikeG.gpkg`. The plots below show the official route geometry (not participant traces), the route distance from the GPKG, the mean filtered-path participant speed from the earlier cell, and the duration implied by `distance / speed`. This is compared with the mean key.csv phase duration.


In [ ]:
# ===========================================================================
# Official route geometry from Experiment path + route-distance / speed / duration check
# ===========================================================================

import geopandas as gpd
from shapely import force_2d, get_parts
from pyproj import Geod

ROUTE_DIR = Path(r'C:\Users\pandya\Documents\Github\docker\ExpData\scripts\Experiment path')
geod = Geod(ellps='WGS84')
route_phase_order = ['WalkG', 'BikeG', 'WalkU', 'BikeU']
all_route_phases = ['WalkG', 'BikeG', 'WalkU', 'BikeU', 'Tram']


def load_route_coords(phase):
    gpkg_path = ROUTE_DIR / f'{phase}.gpkg'
    gdf = gpd.read_file(gpkg_path)
    gdf = gdf.to_crs(4326)
    geom = force_2d(gdf.geometry.iloc[0])
    if geom.geom_type.startswith('Multi'):
        coords = []
        for part in get_parts(geom):
            pts = list(part.coords)
            if coords and coords[-1] == pts[0]:
                coords.extend(pts[1:])
            else:
                coords.extend(pts)
    else:
        coords = list(geom.coords)
    return coords


def route_distance_km(coords):
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    return geod.line_length(lons, lats) / 1000.0


route_summary_rows = []
for phase in all_route_phases:
    coords = load_route_coords(phase)
    dist_km = route_distance_km(coords)
    avg_speed_kmh = np.nan
    avg_key_duration_min = np.nan
    avg_derived_duration_min = np.nan
    if phase in phase_order:
        gps_dict = phase_gps_filtered[phase]
        speeds = []
        durs = []
        for pid in participant_ids:
            if pid not in gps_dict:
                continue
            pid_int = int(pid[1:])
            if (pid_int, phase) not in phase_windows:
                continue
            df = gps_dict[pid]
            path_km = cumulative_path_km(df)
            start_ts, end_ts = phase_windows[(pid_int, phase)]
            dur_h = (end_ts - start_ts).total_seconds() / 3600.0
            if dur_h > 0:
                speeds.append(path_km / dur_h)
                durs.append(dur_h * 60.0)
        avg_speed_kmh = float(np.nanmean(speeds)) if speeds else np.nan
        avg_key_duration_min = float(np.nanmean(durs)) if durs else np.nan
        if pd.notna(avg_speed_kmh) and avg_speed_kmh > 0:
            avg_derived_duration_min = dist_km / avg_speed_kmh * 60.0
    route_summary_rows.append({
        'Phase': phase,
        'route_distance_km': round(dist_km, 3),
        'route_start_lon': round(coords[0][0], 6),
        'route_start_lat': round(coords[0][1], 6),
        'route_end_lon': round(coords[-1][0], 6),
        'route_end_lat': round(coords[-1][1], 6),
        'mean_participant_speed_kmh': round(avg_speed_kmh, 3) if pd.notna(avg_speed_kmh) else np.nan,
        'mean_key_duration_min': round(avg_key_duration_min, 2) if pd.notna(avg_key_duration_min) else np.nan,
        'duration_from_route_distance_and_speed_min': round(avg_derived_duration_min, 2) if pd.notna(avg_derived_duration_min) else np.nan,
        'duration_difference_min': round(avg_derived_duration_min - avg_key_duration_min, 2) if pd.notna(avg_derived_duration_min) and pd.notna(avg_key_duration_min) else np.nan,
    })

route_summary_df = pd.DataFrame(route_summary_rows)

fig, axes = plt.subplots(2, 2, figsize=(15, 12), constrained_layout=True)
axes = axes.ravel()
for ax, phase in zip(axes, route_phase_order):
    coords = load_route_coords(phase)
    lons = [c[0] for c in coords]
    lats = [c[1] for c in coords]
    info = route_summary_df[route_summary_df['Phase'] == phase].iloc[0]
    ax.plot(lons, lats, color=PHASE_COLORS[phase], linewidth=2.2)
    ax.scatter(lons[0], lats[0], color='black', s=30, marker='o', zorder=5)
    ax.scatter(lons[-1], lats[-1], color='black', s=36, marker='s', zorder=5)
    ax.set_title(phase, fontsize=11, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.grid(True, alpha=0.3)
    ax.set_aspect('equal', adjustable='datalim')
    ax.text(
        0.5, 0.14,
        f"route distance = {info['route_distance_km']:.2f} km\n"
        f"mean speed = {info['mean_participant_speed_kmh']:.2f} km/h\n"
        f"derived duration = {info['duration_from_route_distance_and_speed_min']:.2f} min\n"
        f"mean key duration = {info['mean_key_duration_min']:.2f} min",
        transform=ax.transAxes,
        fontsize=ax.xaxis.label.get_size(),
        fontweight='bold',
        ha='center', va='center',
        bbox=dict(facecolor='white', edgecolor='black', alpha=0.82, boxstyle='round,pad=0.3')
    )

fig.suptitle('Official phase routes from Experiment path', fontsize=14, fontweight='bold')
plt.show()

# Check whether BikeG and Tram raw files continued recording outside the labelled phase windows.
continuation_rows = []
for phase in ['BikeG', 'Tram']:
    for pid in sorted({pid for (pid, ph) in raw_data if ph == phase}):
        path = RAW_UCM_DIR / f'P{pid}_{phase}.csv'
        df_full = read_raw_ucm_csv(path)
        if df_full is None or (pid, phase) not in phase_windows:
            continue
        start_ts, end_ts = phase_windows[(pid, phase)]
        offset_h = (start_ts - df_full['GPS_time'].iloc[0]).total_seconds() / 3600.0
        if 1.5 < offset_h < 2.5:
            df_full['GPS_time'] = df_full['GPS_time'] + pd.Timedelta(hours=2)
        continuation_rows.append({
            'ParticipantID': f'P{pid}',
            'Phase': phase,
            'rows_before_phase_window': int((df_full['GPS_time'] < start_ts).sum()),
            'rows_after_phase_window': int((df_full['GPS_time'] > end_ts).sum()),
            'total_rows_in_raw_file': int(len(df_full)),
        })
continuation_df = pd.DataFrame(continuation_rows)

print('Official route summary (including Tram endpoint):')
display(route_summary_df)
print('Raw-file continuation beyond labelled BikeG/Tram phase windows:')
display(continuation_df)
print('Interpretation: BikeG uses the finalized shortened common green route from the GPKG. Tram endpoint is taken from Tram.gpkg. Raw BikeG/Tram files often continue recording before and/or after the labelled phase window, but the analysis uses the clipped phase window only.')



In [ ]:
# ===========================================================================
# Official route timing using Experiment path GPKG files
# Uses: C:/Users/pandya/Documents/Github/docker/ExpData/scripts/Experiment path
# Uses phase times from: C:/Users/pandya/Documents/Github/docker/Paper3_Github/output/key.csv
# Reads staged raw participant phase files from:
# C:/Users/pandya/Documents/Github/docker/Paper3_Github/rawdata/02_ucm
# Raw GPS is first clipped to the exact key2 phase window, then HDOP / IO filtering is applied,
# then only points within a 50 m buffer around the official phase GPKG are kept and shown.
# ===========================================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from osgeo import ogr

RAW_UCM_DIR = Path(r'C:\Users\pandya\Documents\Github\docker\Paper3_Github\rawdata\02_ucm')
ROUTE_DIR = Path(r'C:\Users\pandya\Documents\Github\docker\ExpData\scripts\Experiment path')
KEY_PATH = Path(r'C:\Users\pandya\Documents\Github\docker\Paper3_Github\output\key.csv')
BUFFER_M = 50.0

def parse_ucm_columns(path: Path):
    with open(path, 'r', errors='replace', encoding='utf-8') as f:
        for line in f:
            stripped = line.lstrip('# ').strip()
            if stripped.startswith('GPS_time'):
                return [c.strip() for c in stripped.split(',')]
    return None

def read_raw_ucm_csv(path: Path):
    col_names = parse_ucm_columns(path)
    if col_names is None:
        return None
    df = pd.read_csv(path, comment='#', header=None, names=col_names, low_memory=False)
    df['GPS_time'] = pd.to_datetime(df['GPS_time'], errors='coerce')
    for col in ['GPS_lat', 'GPS_lon', 'GPS_HDOP', 'IO_flag']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    df = df.dropna(subset=['GPS_time', 'GPS_lat', 'GPS_lon']).sort_values('GPS_time').reset_index(drop=True)
    return df

def flag_bad_gps(df):
    bad = np.zeros(len(df), dtype=bool)
    if 'GPS_HDOP' in df.columns:
        bad |= df['GPS_HDOP'].to_numpy() > 5
    if 'IO_flag' in df.columns:
        bad |= df['IO_flag'].to_numpy() == 9
    bad[1:] |= bad[:-1].copy()
    return bad

def find_phase_csv(pid, phase):
    direct = RAW_UCM_DIR / pid / phase / 'ucm' / 'data.csv'
    if direct.exists():
        return direct
    matches = list((RAW_UCM_DIR / pid).glob(f'{phase}/ucm/data.csv'))
    return matches[0] if matches else None

def geometry_xy_segments(geom):
    name = geom.GetGeometryName().upper()
    segments = []
    if 'LINESTRING' in name and 'MULTI' not in name:
        pts = geom.GetPoints()
        if len(pts) >= 2:
            segments.append(([p[0] for p in pts], [p[1] for p in pts]))
    elif 'MULTILINESTRING' in name or name == 'GEOMETRYCOLLECTION':
        for i in range(geom.GetGeometryCount()):
            segments.extend(geometry_xy_segments(geom.GetGeometryRef(i)))
    return segments

def load_route_segments(phase):
    gpkg_path = ROUTE_DIR / f'{phase}.gpkg'
    ds = ogr.Open(str(gpkg_path))
    layer = ds.GetLayer(0)
    segments = []
    for feat in layer:
        geom_ref = feat.GetGeometryRef()
        if geom_ref is not None:
            segments.extend(geometry_xy_segments(geom_ref.Clone()))
    return segments

def lonlat_to_local_m(lon, lat, lon0=4.35, lat0=50.85):
    x = (lon - lon0) * 111320.0 * np.cos(np.radians(lat0))
    y = (lat - lat0) * 110540.0
    return x, y

def point_segment_distance_m(px, py, x1, y1, x2, y2):
    dx = x2 - x1
    dy = y2 - y1
    if dx == 0 and dy == 0:
        return float(np.hypot(px - x1, py - y1))
    t = ((px - x1) * dx + (py - y1) * dy) / (dx * dx + dy * dy)
    t = max(0.0, min(1.0, t))
    proj_x = x1 + t * dx
    proj_y = y1 + t * dy
    return float(np.hypot(px - proj_x, py - proj_y))

def build_metric_route_segments(route_segments_xy):
    segments_m = []
    for xs, ys in route_segments_xy:
        pts_m = [lonlat_to_local_m(x, y) for x, y in zip(xs, ys)]
        for (x1, y1), (x2, y2) in zip(pts_m[:-1], pts_m[1:]):
            segments_m.append((x1, y1, x2, y2))
    return segments_m

def point_within_buffer_m(lat, lon, route_segments_m, buffer_m=BUFFER_M):
    if not route_segments_m:
        return False
    px, py = lonlat_to_local_m(lon, lat)
    min_dist = min(point_segment_distance_m(px, py, x1, y1, x2, y2) for x1, y1, x2, y2 in route_segments_m)
    return min_dist <= buffer_m

def build_phase_windows_from_key(key_path):
    key = pd.read_csv(key_path)
    windows = {}
    for _, row in key.iterrows():
        pid = int(row['Participant_ID'])
        date_str = pd.to_datetime(str(row['Date']) + '-2025', format='%d-%b-%Y').strftime('%Y-%m-%d')
        for phase in ['WalkG', 'BikeG', 'WalkU', 'BikeU', 'Tram']:
            s_col = f'{phase}_start'
            e_col = f'{phase}_end'
            if s_col not in row or e_col not in row:
                continue
            if pd.isna(row[s_col]) or pd.isna(row[e_col]):
                continue
            start = pd.Timestamp(f'{date_str} {row[s_col]}') + pd.Timedelta(hours=2)
            end = pd.Timestamp(f'{date_str} {row[e_col]}') + pd.Timedelta(hours=2)
            if end < start:
                end += pd.Timedelta(days=1)
            windows[(pid, phase)] = (start, end)
    return windows

phase_windows = build_phase_windows_from_key(KEY_PATH)
phases = ['WalkG', 'BikeG', 'WalkU', 'BikeU', 'Tram']
rows = []
fig = plt.figure(figsize=(16, 24))
grid = GridSpec(len(phases), 2, width_ratios=[2.8, 1.2], hspace=0.35, wspace=0.15, figure=fig)

for row_i, phase in enumerate(phases):
    route_segments_xy = load_route_segments(phase)
    route_segments_m = build_metric_route_segments(route_segments_xy)
    ax_map = fig.add_subplot(grid[row_i, 0])
    ax_tbl = fig.add_subplot(grid[row_i, 1])
    ax_tbl.axis('off')
    phase_label_points = []

    first_seg = True
    for xs, ys in route_segments_xy:
        ax_map.plot(xs, ys, color='black', linestyle=':', linewidth=2.3, alpha=0.9,
                    label=f'Official route ({int(BUFFER_M)} m buffer)' if first_seg else None)
        first_seg = False

    table_rows = []
    for pid_dir in sorted(RAW_UCM_DIR.glob('P*'), key=lambda p: int(p.name[1:])):
        pid = pid_dir.name
        pid_int = int(pid[1:])
        csv_path = find_phase_csv(pid, phase)
        if csv_path is None or (pid_int, phase) not in phase_windows:
            continue

        key_start, key_end = phase_windows[(pid_int, phase)]
        df = read_raw_ucm_csv(csv_path)
        if df is None or len(df) < 2:
            continue

        offset_h = (key_start - df['GPS_time'].iloc[0]).total_seconds() / 3600.0
        if 1.5 < offset_h < 2.5:
            df['GPS_time'] = df['GPS_time'] + pd.Timedelta(hours=2)

        df = df[(df['GPS_time'] >= key_start) & (df['GPS_time'] <= key_end)].copy()
        if len(df) >= 2:
            df = df.loc[~flag_bad_gps(df)].copy()
        mask = np.array([point_within_buffer_m(lat, lon, route_segments_m, buffer_m=BUFFER_M)
                         for lat, lon in zip(df['GPS_lat'], df['GPS_lon'])]) if len(df) else np.array([], dtype=bool)
        df_match = df.loc[mask].copy() if len(df) else df.copy()

        color = PARTICIPANT_COLORS.get(pid, '#555555')
        n_points = int(len(df_match))
        if n_points >= 2:
            ax_map.plot(df_match['GPS_lon'], df_match['GPS_lat'], color=color, linewidth=1.4, alpha=0.95)
            start_lon = float(df_match['GPS_lon'].iloc[0])
            start_lat = float(df_match['GPS_lat'].iloc[0])
            end_lon = float(df_match['GPS_lon'].iloc[-1])
            end_lat = float(df_match['GPS_lat'].iloc[-1])
            ax_map.scatter(start_lon, start_lat, color=color, s=26, marker='o', zorder=5)
            ax_map.scatter(end_lon, end_lat, color=color, s=32, marker='s', zorder=5)
            phase_label_points.append({
                'pid': pid, 'kind': 'start', 'lon': start_lon, 'lat': start_lat,
                'text': key_start.strftime('%H:%M:%S'), 'color': color
            })
            phase_label_points.append({
                'pid': pid, 'kind': 'end', 'lon': end_lon, 'lat': end_lat,
                'text': key_end.strftime('%H:%M:%S'), 'color': color
            })

        table_rows.append([pid, key_start.strftime('%H:%M:%S'), key_end.strftime('%H:%M:%S'), n_points])
        rows.append({
            'Phase': phase,
            'ParticipantID': pid,
            'key_start_time': key_start.strftime('%H:%M:%S'),
            'key_end_time': key_end.strftime('%H:%M:%S'),
            'matched_points_within_50m': n_points,
        })

    ax_map.set_title(f'{phase}: only filtered GPS within key time window and {int(BUFFER_M)} m of route', fontsize=12, fontweight='bold')
    ax_map.set_xlabel('Longitude')
    ax_map.set_ylabel('Latitude')
    ax_map.grid(True, alpha=0.25)
    ax_map.set_aspect('equal', adjustable='datalim')
    if row_i == 0:
        ax_map.legend(loc='upper left', fontsize=8, framealpha=0.9)
    if phase_label_points:
        x_span = max(ax_map.get_xlim()) - min(ax_map.get_xlim())
        y_span = max(ax_map.get_ylim()) - min(ax_map.get_ylim())
        x_off = 0.012 * x_span
        y_off = 0.012 * y_span
        for idx, item in enumerate(sorted(phase_label_points, key=lambda d: (d['lon'], d['lat'], d['pid'], d['kind']))):
            direction = -1 if item['kind'] == 'start' else 1
            stagger = ((idx % 4) - 1.5) * 0.35
            tx = item['lon'] + direction * x_off
            ty = item['lat'] + (stagger * y_off)
            ax_map.annotate(
                item['text'],
                xy=(item['lon'], item['lat']),
                xytext=(tx, ty),
                textcoords='data',
                fontsize=7.5,
                color=item['color'],
                ha='left' if direction > 0 else 'right',
                va='center',
                arrowprops=dict(arrowstyle='-', color=item['color'], lw=0.8, alpha=0.85),
                bbox=dict(facecolor='white', edgecolor=item['color'], alpha=0.7, boxstyle='round,pad=0.15'),
                zorder=6,
            )

    table = ax_tbl.table(
        cellText=table_rows,
        colLabels=['Participant', 'key_start', 'key_end', 'n_pts'],
        loc='center',
        cellLoc='center',
    )
    table.auto_set_font_size(False)
    table.set_fontsize(7.3)
    table.scale(1.0, 1.12)
    ax_tbl.set_title(f'{phase} key2 times', fontsize=12, fontweight='bold')

fig.suptitle(
    f'Official Experiment path GPKGs with participant GPS clipped to key phase time and {int(BUFFER_M)} m route buffer',
    fontsize=14, fontweight='bold', y=0.995
)
plt.show()

times_df = pd.DataFrame(rows)
display(times_df)
print(f'Interpretation: this cell uses key.csv phase times, clips raw GPS to those exact times first, applies HDOP/IO filtering, and then only keeps/shows points within {int(BUFFER_M)} m of the official route.')
